In [1]:
!pip install -U langchain langchain-community langchain-core langchain-classic transformers==4.52.4 pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 37.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate

import torch
import re

/tmp/ipykernel_58/3207174176.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [4]:
def generate_text(prompt, max_new_tokens=3000):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [5]:
full_name_schema = ResponseSchema(
    name="full_name",
    description="The candidate's full name as a string."
)

email_schema = ResponseSchema(
    name="email",
    description="The candidate's email address as a string."
)

education_schema = ResponseSchema(
    name="education",
    description=(
        "A list of education objects. Each object must contain "
        "degree, institution, and year."
    )
)

skills_schema = ResponseSchema(
    name="skills",
    description="A list of the candidate's skills as strings."
)

experience_schema = ResponseSchema(
    name="experience",
    description=(
        "A list of experience objects. Each object must contain "
        "role, company, and years."
    )
)

response_schemas = [
    full_name_schema,
    email_schema,
    education_schema,
    skills_schema,
    experience_schema
]

In [6]:
output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)

format_instructions = output_parser.get_format_instructions()

In [7]:
pdf_path = "/kaggle/input/datasets/andrewsameh20/andrew-sameh-resume/Andrew Sameh Resume.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(f"Number of pages: {len(documents)}")

Number of pages: 2


In [8]:
pdf_text = "\n\n".join(
    document.page_content
    for document in documents
)

print(pdf_text)

Andrew Sameh  
SOFTWARE ENGINEER | ARTIFICIAL INTELLIGENCE & MACHINE LEARNING   
Cairo , Egypt | +20 10 1966 4833 | andrewsamehshaker@gmail.com | 
www.linkedin.com/in/andrew-sameh-software-engineer | https://github.com/AndrewSameh7  
Professional Summary 
Software Engineering graduate with a Very Good overall grade and an Excellent-rated graduation project, 
specializing in Artificial Intelligence and Machine Learning. Hands-on experience in Deep Learning, Computer 
Vision, NLP, and AI for Robotics through internships, applied projects, and an AI Hackathon 2026 2nd-place 
achievement. Experienced in developing AI solutions using YOLOv8, LSTM, and Bi-LSTM for object detection, 
forecasting, sentiment analysis, and intelligent chatbot systems, with a growing foundation in LLMs and 
Transformer Models through Hugging Face training. Strong foundation in Python, C++, Software Engineering, 
and algorithmic problem-solving, demonstrated through 3-time ECPC participation with Honorable Mention

In [9]:
print(pdf_text[:5000])

Andrew Sameh  
SOFTWARE ENGINEER | ARTIFICIAL INTELLIGENCE & MACHINE LEARNING   
Cairo , Egypt | +20 10 1966 4833 | andrewsamehshaker@gmail.com | 
www.linkedin.com/in/andrew-sameh-software-engineer | https://github.com/AndrewSameh7  
Professional Summary 
Software Engineering graduate with a Very Good overall grade and an Excellent-rated graduation project, 
specializing in Artificial Intelligence and Machine Learning. Hands-on experience in Deep Learning, Computer 
Vision, NLP, and AI for Robotics through internships, applied projects, and an AI Hackathon 2026 2nd-place 
achievement. Experienced in developing AI solutions using YOLOv8, LSTM, and Bi-LSTM for object detection, 
forecasting, sentiment analysis, and intelligent chatbot systems, with a growing foundation in LLMs and 
Transformer Models through Hugging Face training. Strong foundation in Python, C++, Software Engineering, 
and algorithmic problem-solving, demonstrated through 3-time ECPC participation with Honorable Mention

In [10]:
resume_extraction_template = """
You are an HR candidate profile parser.

Your task is to extract structured candidate information from the provided resume text.

IMPORTANT:
You MUST return ALL five required fields.
Never omit any field.

The required JSON fields are exactly:

1. "full_name"
   - The candidate's full name as a string.

2. "email"
   - The candidate's email address as a string.

3. "education"
   - A list of objects.
   - Each object must contain:
     "degree", "institution", "year"

4. "skills"
   - A list of strings.
   - If no skills are found, return an empty list [].

5. "experience"
   - A list of objects.
   - Each object must contain:
     "role", "company", "years"
   - If no experience is found, return an empty list [].

Do not invent information.

If information is missing:
- For missing string values, use an empty string "".
- For missing list values, use [].

You MUST include all five fields in the final JSON.

{format_instructions}

Resume text:
{user_input}
"""

In [11]:
prompt = PromptTemplate(
    template=resume_extraction_template,
    input_variables=[
        "user_input",
        "format_instructions"
    ]
).format(
    user_input=pdf_text,
    format_instructions=format_instructions
)

In [12]:
response = generate_text(
    prompt,
    max_new_tokens=3000
)

print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



```json
{
	"full_name": "Andrew Sameh",
	"email": "andrewsamehshaker@gmail.com",
	"education": [
		{
			"degree": "B.Sc.",
			"institution": "King Salman International University (KSIU)",
			"year": "2021 –2025"
		}
	],
	"skills": [
		"Python",
		"C++",
		"Object-oriented programming (OOP)",
		"Machine Learning",
		"Classification",
		"Regression",
		"Clustering",
		"Feature Engineering",
		"Model Evaluation",
		"Deep Learning",
		"Convolutional Neural Network (CNN)",
		"Recurrent Neural Network (RNN)",
		"Long Short-Term Memory (LSTM)",
		"Bidirectional Long Short-Term Memory (Bi-LSTM)",
		"Neural Networks",
		"Computer Vision",
		"OpenCV",
		"Object Detection",
		"Image Processing",
		"Text Preprocessing",
		"Tokenization",
		"Sentiment Analysis",
		"Generative AI",
		"LLMs",
		"RAG",
		"Embeddings",
		"Vector Databases",
		"Prompt Engineering",
		"LLM APIs",
		"Data Structures & Algorithms",
		"Software Engineering",
		"Databases",
		"System Analysis & Design",
		"Data",
		"Pandas"

In [13]:
def extract_json_block(text):
    fenced_pattern = r'```json\s*(.*?)\s*```'

    matches = re.findall(
        fenced_pattern,
        text,
        re.DOTALL | re.IGNORECASE
    )

    if matches:
        return matches[-1].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        return text[start:end + 1].strip()

    raise ValueError("No JSON object found in model response.")

In [14]:
json_text = extract_json_block(response)

print(json_text)

{
	"full_name": "Andrew Sameh",
	"email": "andrewsamehshaker@gmail.com",
	"education": [
		{
			"degree": "B.Sc.",
			"institution": "King Salman International University (KSIU)",
			"year": "2021 –2025"
		}
	],
	"skills": [
		"Python",
		"C++",
		"Object-oriented programming (OOP)",
		"Machine Learning",
		"Classification",
		"Regression",
		"Clustering",
		"Feature Engineering",
		"Model Evaluation",
		"Deep Learning",
		"Convolutional Neural Network (CNN)",
		"Recurrent Neural Network (RNN)",
		"Long Short-Term Memory (LSTM)",
		"Bidirectional Long Short-Term Memory (Bi-LSTM)",
		"Neural Networks",
		"Computer Vision",
		"OpenCV",
		"Object Detection",
		"Image Processing",
		"Text Preprocessing",
		"Tokenization",
		"Sentiment Analysis",
		"Generative AI",
		"LLMs",
		"RAG",
		"Embeddings",
		"Vector Databases",
		"Prompt Engineering",
		"LLM APIs",
		"Data Structures & Algorithms",
		"Software Engineering",
		"Databases",
		"System Analysis & Design",
		"Data",
		"Pandas",
		"NumP

In [15]:
output_data = output_parser.parse(json_text)

print(output_data)

{'full_name': 'Andrew Sameh', 'email': 'andrewsamehshaker@gmail.com', 'education': [{'degree': 'B.Sc.', 'institution': 'King Salman International University (KSIU)', 'year': '2021 –2025'}], 'skills': ['Python', 'C++', 'Object-oriented programming (OOP)', 'Machine Learning', 'Classification', 'Regression', 'Clustering', 'Feature Engineering', 'Model Evaluation', 'Deep Learning', 'Convolutional Neural Network (CNN)', 'Recurrent Neural Network (RNN)', 'Long Short-Term Memory (LSTM)', 'Bidirectional Long Short-Term Memory (Bi-LSTM)', 'Neural Networks', 'Computer Vision', 'OpenCV', 'Object Detection', 'Image Processing', 'Text Preprocessing', 'Tokenization', 'Sentiment Analysis', 'Generative AI', 'LLMs', 'RAG', 'Embeddings', 'Vector Databases', 'Prompt Engineering', 'LLM APIs', 'Data Structures & Algorithms', 'Software Engineering', 'Databases', 'System Analysis & Design', 'Data', 'Pandas', 'NumPy', 'Matplotlib', 'EDA', 'Data Preprocessing', 'Time-Series Analysis', 'Frameworks', 'TensorFlow

In [16]:
print("Full Name:")
print(output_data["full_name"])

print("\nEmail:")
print(output_data["email"])

print("\nEducation:")
for education in output_data["education"]:
    print(education)

print("\nSkills:")
for skill in output_data["skills"]:
    print("-", skill)

print("\nExperience:")
for experience in output_data["experience"]:
    print(experience)

Full Name:
Andrew Sameh

Email:
andrewsamehshaker@gmail.com

Education:
{'degree': 'B.Sc.', 'institution': 'King Salman International University (KSIU)', 'year': '2021 –2025'}

Skills:
- Python
- C++
- Object-oriented programming (OOP)
- Machine Learning
- Classification
- Regression
- Clustering
- Feature Engineering
- Model Evaluation
- Deep Learning
- Convolutional Neural Network (CNN)
- Recurrent Neural Network (RNN)
- Long Short-Term Memory (LSTM)
- Bidirectional Long Short-Term Memory (Bi-LSTM)
- Neural Networks
- Computer Vision
- OpenCV
- Object Detection
- Image Processing
- Text Preprocessing
- Tokenization
- Sentiment Analysis
- Generative AI
- LLMs
- RAG
- Embeddings
- Vector Databases
- Prompt Engineering
- LLM APIs
- Data Structures & Algorithms
- Software Engineering
- Databases
- System Analysis & Design
- Data
- Pandas
- NumPy
- Matplotlib
- EDA
- Data Preprocessing
- Time-Series Analysis
- Frameworks
- TensorFlow
- Keras
- PyTorch
- Scikit-learn
- Tools
- Git
- GitHub

In [17]:
required_fields = [
    "full_name",
    "email",
    "education",
    "skills",
    "experience"
]

for field in required_fields:
    assert field in output_data

assert isinstance(output_data["education"], list)
assert isinstance(output_data["skills"], list)
assert isinstance(output_data["experience"], list)

print("PDF HR Candidate Profile Parser test passed successfully!")

PDF HR Candidate Profile Parser test passed successfully!
